In [2]:
%pip install crunch-cli --upgrade --quiet --progress-bar off
!crunch setup-notebook structural-break-real-time mrJxEtUHQjjR3I6YxoLYAqKm

crunch-cli, version 11.9.0
main.py: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/submissions/68419/main.py (66287 bytes)
notebook.ipynb: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/submissions/68419/notebook.ipynb (107492 bytes)
requirements.txt: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/submissions/68419/requirements.txt (203 bytes)
resources/model.joblib: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/models/61063/model.joblib (1050782 bytes)
data/X_train.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/234/X_train.parquet (218514418 bytes)
data/X_test.reduced.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/234/X_test.reduced.parquet (2587435 bytes)
data/y_train.parquet: download from https:crunchdao--competition--produ

In [3]:
import math
import os
from typing import Iterable, List, Optional, Tuple

# Import your dependencies.
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

In [4]:
import crunch

# Load the Crunch Toolings (data loader, local tester, submitter).
crunch_tools = crunch.load_notebook()

loaded inline runner with module: <module '__main__'>

cli version: 11.9.0
available ram: 12.67 gb
available cpu: 2 core
----


In [5]:
# Load the data.
train_data, test_data = crunch_tools.load_data()

data/X_train.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/234/X_train.parquet (218514418 bytes)
data/X_train.parquet: already exists, file length match
data/X_test.reduced.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/234/X_test.reduced.parquet (2587435 bytes)
data/X_test.reduced.parquet: already exists, file length match
data/y_train.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/234/y_train.parquet (8356193 bytes)
data/y_train.parquet: already exists, file length match
data/y_test.reduced.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/234/y_test.reduced.parquet (106299 bytes)
data/y_test.reduced.parquet: already exists, file length match
data/y_test_index.reduced.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws

### A single training series

Let us look at one training series with a break, to get a feel for the
data.


In [15]:
def train(datasets, model_directory_path):
    import numpy as np, math, os, gc
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import roc_auc_score

    MODEL_VERSION = "fast-linear-v1"
    mpath = os.path.join(model_directory_path, "model.joblib")

    if os.path.exists(mpath):
        try:
            bundle = joblib.load(mpath)
            if bundle.get("version") == MODEL_VERSION:
                print(f"pre-trained model found ({MODEL_VERSION}) — skipping training")
                return
        except Exception as e:
            print(f"existing model unreadable ({e}) — retraining")

    datasets = list(datasets)
    rng = np.random.RandomState(42)
    perm = rng.permutation(len(datasets))
    datasets = [datasets[i] for i in perm]
    n_val = len(datasets) // 5
    val_ds, tr_ds = datasets[:n_val], datasets[n_val:]

    print(f"Training on {len(tr_ds)} series, validating on {len(val_ds)}")

    def build_features(ds):
        all_feats, all_labels, all_sids, all_ts = [], [], [], []
        for sid, (dataset_id, x_hist, x_online, tau) in enumerate(ds):
            h = np.asarray(x_hist, dtype=np.float64)
            mu_h = float(h.mean())
            sd_h = max(float(h.std(ddof=1)), 1e-9)
            zh = (h - mu_h) / sd_h
            n = len(zh)

            if n >= 50:
                y = zh[2:]
                X = np.column_stack([zh[1:-1], zh[:-2]])
                XtX = X.T @ X + 1e-6 * np.eye(2)
                phi = np.linalg.solve(XtX, X.T @ y)
                p1, p2 = float(phi[0]), float(phi[1])
                rv = max(float((y - X @ phi).var()), 1e-6)
            else:
                p1, p2, rv = 0.0, 0.0, 1.0

            z1 = float(zh[-1])
            z2 = float(zh[-2]) if n >= 2 else 0.0

            cusum_pos = 0.0
            cusum_neg = 0.0
            t = 0
            m = 0.0
            m2 = 0.0
            ew_var = 1.0
            buf = []
            q_hist = np.quantile(zh, np.linspace(0.05, 0.95, 19))

            for t_step, point in enumerate(x_online):
                t += 1
                z = (float(point) - mu_h) / sd_h
                pred = p1 * z1 + p2 * z2
                u = (z - pred) / math.sqrt(rv)
                z2 = z1
                z1 = z

                cusum_pos = max(0.0, cusum_pos + u - 0.5)
                cusum_neg = max(0.0, cusum_neg - u - 0.5)
                cusum = max(cusum_pos, cusum_neg)
                d1 = math.tanh(cusum / math.sqrt(t) / 2.0)

                delta = u - m
                m += delta / t
                delta2 = u - m
                m2 += delta * delta2
                var_online = m2 / t if t > 1 else 1.0
                var_ratio = abs(math.log(max(var_online, 1e-12)))
                ew_var = 0.9 * ew_var + 0.1 * var_ratio
                d2 = math.tanh(ew_var / 1.5)

                buf.append(u)
                if len(buf) > 50:
                    buf.pop(0)
                if len(buf) >= 10:
                    arr = np.array(buf)
                    q_win = np.quantile(arr, np.linspace(0.05, 0.95, 19))
                    ks = float(np.max(np.abs(q_win - q_hist)))
                else:
                    ks = 0.0
                d3 = math.tanh(ks / 0.5)

                t_norm = t / 1000.0
                log_len = math.log(len(q_hist) * 5 + 1)
                log_sd = math.log(sd_h + 1e-9)

                all_feats.append([d1, d2, d3, t_norm, log_len, log_sd])
                label = 1 if (tau is not None and t_step >= tau) else 0
                all_labels.append(label)
                all_sids.append(sid)
                all_ts.append(t_step)

        return (
            np.array(all_feats, dtype=np.float32),
            np.array(all_labels, dtype=np.float32),
            np.array(all_sids, dtype=np.int32),
            np.array(all_ts, dtype=np.int32),
        )

    cache = "/tmp/fast_linear_features.npz"
    if os.path.exists(cache):
        d = np.load(cache)
        X_tr, y_tr, sid_tr, t_tr = d["X_tr"], d["y_tr"], d["sid_tr"], d["t_tr"]
        X_va, y_va, sid_va, t_va = d["X_va"], d["y_va"], d["sid_va"], d["t_va"]
        print("features loaded from cache")
    else:
        print("building training features...")
        X_tr, y_tr, sid_tr, t_tr = build_features(tr_ds)
        print("building validation features...")
        X_va, y_va, sid_va, t_va = build_features(val_ds)
        np.savez(cache, X_tr=X_tr, y_tr=y_tr, sid_tr=sid_tr, t_tr=t_tr,
                 X_va=X_va, y_va=y_va, sid_va=sid_va, t_va=t_va)
        print("features cached")

    lin = LogisticRegression(max_iter=1000, C=1.0)
    lin.fit(X_tr, y_tr)

    p_va = lin.predict_proba(X_va)[:, 1]

    def eval_ts_auc(y, preds, ts):
        num = den = 0.0
        for t in np.unique(ts):
            m = ts == t
            npos = int(y[m].sum())
            nneg = int((1 - y[m]).sum())
            if npos == 0 or nneg == 0:
                continue
            w = npos * nneg
            num += w * roc_auc_score(y[m], preds[m])
            den += w
        return num / den if den > 0 else 0.5

    auc = eval_ts_auc(y_va, p_va, t_va)
    print(f"Validation TS-AUC: {auc:.4f}")

    joblib.dump({
        "version": MODEL_VERSION,
        "w": lin.coef_[0].astype(np.float32),
        "b": float(lin.intercept_[0]),
    }, mpath)
    print(f"Model saved to {mpath}")

    del X_tr, y_tr, X_va, y_va
    gc.collect()

### The `infer()` function

`infer()` is a **generator**. It must:

1. Load any model saved by `train()`.
2. `yield` once, with no value, to signal readiness to the runner.
3. For each test series `(x_historical, x_online)`:
    - Pre-compute whatever static summaries you need from `x_historical` (it is given in full and cheap to scan once).
    - Loop over the points of `x_online`. After each point, `yield` a `float` in $[0, 1]$ -- **exactly one yield per online point**, in order.

<br />

> **‼ Important**
> ---
> Both the outer `datasets` iterable and each `x_online` can be iterated **only once** in the cloud environment. <br />
> You must produce the score for the current observation before the next one is released.

### The streaming EWMA detector in code

We keep three running scalars per series:

- `mu_ewma`: the EWMA of the online values (tracks the current local mean).
- `n_eff`: the effective sample size of the EWMA (grows as more points arrive, bounded by `1 / (1 - alpha)`).
- `t`: the step index, for optional diagnostics.

At each new observation `x_t` we update these in O(1) and emit the score.

In [16]:
def infer(datasets, model_directory_path):
    import numpy as np, math, os, joblib

    mpath = os.path.join(model_directory_path, "model.joblib")
    bundle = joblib.load(mpath)
    w = bundle["w"]
    b = bundle["b"]

    def sigmoid(x):
        if x >= 0:
            z = math.exp(-x)
            return 1.0 / (1.0 + z)
        else:
            z = math.exp(x)
            return z / (1.0 + z)

    yield

    for x_historical, x_online in datasets:
        h = np.asarray(x_historical, dtype=np.float64)
        mu_h = float(h.mean())
        sd_h = max(float(h.std(ddof=1)), 1e-9)
        zh = (h - mu_h) / sd_h
        n = len(zh)

        if n >= 50:
            y = zh[2:]
            X = np.column_stack([zh[1:-1], zh[:-2]])
            XtX = X.T @ X + 1e-6 * np.eye(2)
            phi = np.linalg.solve(XtX, X.T @ y)
            p1, p2 = float(phi[0]), float(phi[1])
            rv = max(float((y - X @ phi).var()), 1e-6)
        else:
            p1, p2, rv = 0.0, 0.0, 1.0

        z1 = float(zh[-1])
        z2 = float(zh[-2]) if n >= 2 else 0.0

        cusum_pos = 0.0
        cusum_neg = 0.0
        t = 0
        m = 0.0
        m2 = 0.0
        ew_var = 1.0
        buf = []
        q_hist = np.quantile(zh, np.linspace(0.05, 0.95, 19))
        peak = 0.0

        for point in x_online:
            t += 1
            z = (float(point) - mu_h) / sd_h
            pred = p1 * z1 + p2 * z2
            u = (z - pred) / math.sqrt(rv)
            z2 = z1
            z1 = z

            cusum_pos = max(0.0, cusum_pos + u - 0.5)
            cusum_neg = max(0.0, cusum_neg - u - 0.5)
            cusum = max(cusum_pos, cusum_neg)
            d1 = math.tanh(cusum / math.sqrt(t) / 2.0)

            delta = u - m
            m += delta / t
            delta2 = u - m
            m2 += delta * delta2
            var_online = m2 / t if t > 1 else 1.0
            var_ratio = abs(math.log(max(var_online, 1e-12)))
            ew_var = 0.9 * ew_var + 0.1 * var_ratio
            d2 = math.tanh(ew_var / 1.5)

            buf.append(u)
            if len(buf) > 50:
                buf.pop(0)
            if len(buf) >= 10:
                arr = np.array(buf)
                q_win = np.quantile(arr, np.linspace(0.05, 0.95, 19))
                ks = float(np.max(np.abs(q_win - q_hist)))
            else:
                ks = 0.0
            d3 = math.tanh(ks / 0.5)

            d4 = t / 1000.0
            d5 = math.log(len(q_hist) * 5 + 1)
            d6 = math.log(sd_h + 1e-9)

            logit = w[0]*d1 + w[1]*d2 + w[2]*d3 + w[3]*d4 + w[4]*d5 + w[5]*d6 + b
            score = sigmoid(logit)

            if score > peak:
                peak = score

            yield peak

### Parallelism

If your model is capable of running in parallel, you should try enabling the parallelism mechanism. You can [learn more in the documentation](https://docs.crunchdao.com/competitions/competitions/structural-break-real-time#parallelism).

But the TL;DR is:
- Your model will be run N times, with each call happening in a different process (rather than a thread).

- To avoid concurrency issues, do not write file from the `infer()` function.

- Make sure you don't use too many resources. <br />
  - If your model requires 4 GB of RAM and you want a parallelism of 6, the machine will need at least 24 GB of RAM (+overhead).
  - The same applies to CPUs: try not to use more than the number of CPUs your machine has (+overhead).

- It makes debugging more difficult: if you need to diagnose a bug, revert to a value of `1`.

In [ ]:
# @crunch/keep:on
INFER_PARALLELISM = 4

# Uncomment this line to run `infer` with maximum parallelism while leaving one CPU core free for other tasks
# INFER_PARALLELISM = os.cpu_count() - 1

# Uncomment this line to run infer without parallelism
# INFER_PARALLELISM = 1

## Local testing

The Crunch CLI ships with a local tester that reproduces the cloud
environment. <br />
It calls `train()` once, then calls `infer()` with the test data, collects the yielded scores, and writes them to `prediction/prediction.parquet`.

This is the same flow the platform will run.

In [9]:
crunch_tools.test(
    # Uncomment to skip re-training each time
    # force_first_train=False,

    # Uncomment to skip the determinism check
    # no_determinism_check=True,
)

12:49:42 
12:49:42 started
12:49:42 running local test
12:49:42 internet access isn't restricted, no check will be done
12:49:42 
12:49:44 executing - command=train


data/X_train.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/234/X_train.parquet (218514418 bytes)
data/X_train.parquet: already exists, file length match
data/X_test.reduced.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/234/X_test.reduced.parquet (2587435 bytes)
data/X_test.reduced.parquet: already exists, file length match
data/y_train.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/234/y_train.parquet (8356193 bytes)
data/y_train.parquet: already exists, file length match
data/y_test.reduced.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/234/y_test.reduced.parquet (106299 bytes)
data/y_test.reduced.parquet: already exists, file length match
data/y_test_index.reduced.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws

13:19:40 executing - command=get_parallelism
[parallelism] `INFER_PARALLELISM` must be at most the number of CPUs (2)
13:19:40 using a parallelism of 2
13:19:40 executing - command=infer
13:19:40 executing - command=infer
Process ForkProcess-1:
Process ForkProcess-2:
13:19:40 [debug] error in data thread: connection closed by client
13:19:40 [debug] error in data thread: connection closed by client
Traceback (most recent call last):
Traceback (most recent call last):
  File "/usr/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/usr/lib/python3.12/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "https://github.com/crunchdao/competitions/raw/refs/heads/master/competitions/structural-break-real-time/scoring/runner.py", line 469, in do_execute
  File "/usr/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/usr/lib/python3.12/multiprocessing/process.py", lin

## Previewing the results

In [10]:
prediction = pd.read_parquet("prediction/prediction.parquet")
prediction.head(10)

FileNotFoundError: [Errno 2] No such file or directory: 'prediction/prediction.parquet'

### Computing TS-AUC locally

Below is a straightforward implementation of the TS-AUC metric: group rows by online time step, compute the AUC cross-sectionally at each step (skipping steps that only have one class), and return the weighted average.

This is exactly what the leaderboard uses, just without the private test set.

In [ ]:
# Load the ground-truth labels supplied with the local tester.
y_test = pd.read_parquet("data/y_test.reduced.parquet")

# Merge predictions with true labels on (id, time).
merged = prediction.merge(
    y_test,
    how="left",
    left_index=True,
    right_index=True,
)

# Add the online step index (0, 1, 2, ...).
merged["time_online"] = merged.groupby("id").cumcount()

# Weighted per-step AUC.
weighted_auc_sum = 0.0
total_weight     = 0.0

for t, group in merged.groupby("time_online"):
    labels = group["target"].values
    scores = group["prediction"].values

    n_pos = int(labels.sum())
    n_neg = int((1 - labels).sum())
    if n_pos == 0 or n_neg == 0:
        continue

    auc_t  = float(roc_auc_score(labels, scores))
    weight = float(n_pos * n_neg)

    weighted_auc_sum += weight * auc_t
    total_weight     += weight

ts_auc = weighted_auc_sum / total_weight if total_weight > 0 else 0.5
print(f"Local TS-AUC: {ts_auc:.4f}")

# Submitting your notebook

To submit your work, you must:

1. Download your notebook (from Colab, Kaggle, or a local copy).
2. Upload it to the competition platform.
3. Create a **run** to validate it against the full test set.

Executing the cell below will take care of everything (only available on Google Colab), or show you how to submit manually.

In [17]:
# @title  {"display-mode":"form", "form-width":"400px"}

# @markdown Describe your changes, then run the cell.
Message = "" # @param {"type":"string","placeholder":"Short description (optional)"}

# ---
# THIS METHOD IS ONLY POSSIBLE ON COLAB.
# RUNNING THIS CELL WILL PROMPT YOU TO USE THE OLD WAY OF SUBMITTING A NOTEBOOK.

crunch_tools.submit(
    message=Message,
)

warning 101d5543: line 1: column 0: nested import: found 3 nested imports in FunctionDef statement
warning a4c0cc77: line 1: column 0: nested import: found 1 nested import in FunctionDef statement


found code file: requirements.txt (158 bytes)


uploading `requirements.txt`:   0%|          | 0.00/158 [00:00<?, ?B/s]

found code file: main.py (10.54 KB)


uploading `main.py`:   0%|          | 0.00/10.3k [00:00<?, ?B/s]

...

uploading `notebook.ipynb`:   0%|          | 0.00/34.3k [00:00<?, ?B/s]

total code size: 45.83 KB
found model file: model.joblib (4 KB)


uploading `model.joblib`:   0%|          | 0.00/3.91k [00:00<?, ?B/s]

total model size: 4 KB
export structural-break-real-time:project/13312/breakyy



---

Next step is to run your submission in the cloud:

### >> https://hub.crunchdao.com/competitions/structural-break-real-time/models/pixelated-anja/breakyy/runs/create?submissionNumber=37

<img alt="Run in the Cloud" src="https://raw.githubusercontent.com/crunchdao/competitions/refs/heads/master/documentation/animations/create-run.gif" height="600px" />
